# RAG with LlamaIndex- The Data Framework

**Module 8 · Lesson 8.6**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- LlamaIndex RAG in 5 Lines - VectorStoreIndex
- Customizing the Pipeline - Chunking, Prompts, top-k
- Chat Engine - Conversational RAG in One Line
- The Data Model + Three-Way Comparison
- LlamaParse - GenAI-Native Document Parsing
- IngestionPipeline - Declarative Processing with Caching

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

## Five Lines, One Afternoon Saved

> 💡 **Info**
>
> Your team's RAG service works. In 8.1 you hand-built it, in 8.5 you rebuilt it with LangChain. Then a new requirement lands: ingest 4,000 messy vendor PDFs - scanned invoices, multi-column contracts, tables inside tables - and answer questions across all of them. With the general-purpose stack that is a week of gluing a PDF parser, a chunker, a caching layer, and a comparison-query router. A colleague reaches for LlamaIndex instead: `VectorStoreIndex.from_documents()` for the pipeline, LlamaParse for the ugly PDFs, an IngestionPipeline that only reprocesses what changed, and a SubQuestionQueryEngine for the cross-document comparisons. It is running by lunch. **That is the trade this lesson teaches:** LlamaIndex is narrower than LangChain, and for the one job of turning documents into answers, narrower is faster, cleaner, and better-equipped.

### 🎯 What you will be able to do after this lesson

- **Build** RAG in ~5 lines with `VectorStoreIndex.from_documents` and `as_query_engine`, on the current `GoogleGenAI` + `gemini-embedding-001` stack.

- **Operate** the pipeline - chunking (`SentenceSplitter`), prompts, top-k, the one-line chat engine, and the caching `IngestionPipeline`.

- **Compare** LlamaIndex's data-specialist tools - LlamaParse, response-synthesis modes, SubQuestion/Router query engines, FunctionAgent - by the problem each solves.

- **Evaluate** when to reach for LlamaIndex vs LangChain, and how to combine them in the production hybrid.

> 📦 **Info**
>
> 🧭 Before you start
> You need Lessons 8.1 (the from-scratch pipeline), 8.3 (embeddings + a vector store), and 8.5 (the LangChain build this shrinks), a Google AI Studio key in `GOOGLE_API_KEY`, and `pip install llama-index llama-index-llms-google-genai llama-index-embeddings-google-genai`. Every block targets **LlamaIndex 0.14.x**: pre-2025 tutorials import the discontinued `Gemini` class and pass a removed `ServiceContext` - both error on the current stack.

## 60-Second Warm-Up: What You Already Know

Three recalls - all load-bearing today. Click a card to check yourself.

> 🍣 **Analogy**
>
> **LangChain is a full workshop; LlamaIndex is a master sushi knife.** The workshop does everything - chains, agents, memory, deployment. The sushi knife does ONE thing, connecting data to an LLM, and for that one thing it makes a cleaner cut than the workshop's general-purpose blade. `VectorStoreIndex.from_documents()` is that single confident cut: a whole fish (your documents) becomes service-ready pieces (chunked, embedded, indexed nodes) in one motion.
> **Where the analogy breaks down:** a sushi knife is useless for anything that is not fish, and LlamaIndex is deliberately narrower than LangChain - for stateful multi-tool agents, checkpointed graphs, and deep observability you still reach for LangChain/LangGraph. They are a knife and a workshop you keep on the same bench, not rivals.

> 💡 **Info**
>
> ⚠️ Misconception: "LlamaIndex is just LangChain with fewer lines" (and "you must pick one")
> Both wrong. Fewer lines is a symptom, not the point: LlamaIndex is a **data specialist**, and the depth shows in tools LangChain has no direct equal for - LlamaParse (GenAI-native PDF parsing), the caching IngestionPipeline, and six response-synthesis modes. And you do not pick one and abandon the other: the 2026 production default is **LlamaIndex for the data layer, LangChain for orchestration**, joined by `QueryEngineTool.as_langchain_tool()`. The anti-pattern to avoid is forcing one framework to do the job the other is built for (see the [run-llama repository](https://github.com/run-llama/llama_index) and the [LlamaParse v2 blog](https://www.llamaindex.ai/blog/introducing-llamaparse-v2-simpler-better-cheaper)).

~5 lines. Data-specialist. LlamaParse, caching ingestion, synthesis modes. **Best for:** RAG-first apps, complex-document ingestion.

~15 lines. Orchestration-specialist. LangGraph, agents, LangSmith. **Best for:** stateful agents, routing, deployment, monitoring.

## Build One: RAG in 5 Lines

Steps 1-4 build RAG the LlamaIndex way: the 5-line express version, customizing chunking and prompts, the one-line chat engine, and the Document-to-Node-to-Index data model plus the three-way framework comparison.

The LlamaIndex data model

---

## 🎯 Concept 1: LlamaIndex RAG in 5 Lines - VectorStoreIndex

### LlamaIndex RAG in 5 Lines - VectorStoreIndex

One call chunks, embeds, and stores. A second call wires retrieve and synthesize.

**Why this is step 1:** it is the whole framework on one screen. The pipeline is the same one you hand-built in 8.1 (60 lines) and rebuilt in 8.5 (15 lines), now about 5 - because `VectorStoreIndex.from_documents(docs)` hides chunking, embedding, and storage behind a single call, and `as_query_engine()` hides retrieval and synthesis - what you called generate in 8.1, turning the retrieved chunks into a written answer - behind another.

> 🔪 **Analogy**
>
> **This is the single confident cut.** A master sushi chef does not saw at the fish; one clean motion turns the whole fish into uniform, service-ready pieces. `from_documents` is that motion - documents in, indexed nodes out, no fumbling with chunkers and embedders.
> **Where the analogy breaks down:** the chef's single cut is final, but LlamaIndex's is configurable - step 2 shows you can re-grind the blade (chunk size, prompt, top-k) when the default cut is wrong for your fish.

How many separate operations does the single call `VectorStoreIndex.from_documents(docs)` actually perform?

- **Settings** is the global config (it replaced the removed `ServiceContext`): set the LLM and embedder once.

- **Documents** are your raw sources.

- **`from_documents`** chunks + embeds + stores in one call.

- **`as_query_engine`** then answers, and `source_nodes` shows what it retrieved.

**📝 `01_llamaindex_rag.py`** - *LlamaIndex*

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install llama-index llama-index-llms-google-genai llama-index-embeddings-google-genai llama-index-embeddings-huggingface sentence-transformers llama-cloud-services python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


#### 💡 What just happened

⚡ What just happened?`VectorStoreIndex.from_documents()` did three things in one call - chunked each document into nodes (a node is LlamaIndex’s word for a chunk plus its metadata - full definition in step 4), embedded each node, and stored the vectors; `as_query_engine()` wired retrieve + synthesize. **Five lines replaced 60 (8.1) and 15 (8.5) - but only because those builds taught you what each hidden step does.** Two facts that break old tutorials: the `Gemini` class is discontinued (use `GoogleGenAI`), and `gemini-2.0-flash` now returns an API error (use `gemini-2.5-flash`). The tradeoff for the terseness: LlamaIndex's opinionated defaults hide control that LangChain exposes - which step 2 hands back.

- Click a document tile to send it down the pipeline: Loader, then Node-split, then Embed, then Index.

- Type or pick a query and watch retrieval light the matching nodes.

- Watch the QueryEngine pull the top-k nodes and synthesize the answer.

- The panel counts nodes and shows which ones were retrieved - the Document-to-Node-to-Index-to-QueryEngine model, live.

Interactive: click tiles, pick a query, watch the retrieval light up. This is what from_documents hides.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Customizing the Pipeline - Chunking, Prompts, top-k

### Customizing the Pipeline - Chunking, Prompts, top-k

Simple by default, fully controllable when the default cut is wrong.

**The default cut is not always right for your fish.** LlamaIndex lets you re-grind every part of the pipeline: the chunker (a `SentenceSplitter` with a `chunk_size` and `chunk_overlap`), the prompt (a grounded `PromptTemplate`), and how many chunks to retrieve (`similarity_top_k`). Of these, chunk size is the single highest-leverage knob - it decides what a "unit of retrieval" even is.

> 🍣 **Analogy**
>
> **Chunk size is how thick you slice.** Slice too thin (small chunks) and each piece is a precise, clean bite - but it may miss the context of the piece beside it. Slice too thick (large chunks) and each piece carries plenty of context - but the flavour you actually wanted is diluted among the rest. The chef picks thickness for the dish; you pick chunk size for the query.
> **Where the analogy breaks down:** a chef slices with no overlap, but chunks can *overlap* - `chunk_overlap` repeats a few tokens at each boundary so a sentence split across two chunks is not lost. There is no sushi equivalent of two slices sharing the same rice.

You shrink `chunk_size` from 512 to 128 tokens. What happens to retrieval?

- **`SentenceSplitter`** as a transformation controls chunk size and overlap.

- **`PromptTemplate`** grounds the answer (`{context_str}` + `{query_str}`).

- **`similarity_top_k`** sets how many chunks feed the answer; metadata tracks sources.

**📝 `02_custom_pipeline.py`** - *LlamaIndex*

In [ ]:
# LLAMAINDEX - a customized pipeline
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.prompts import PromptTemplate
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

Settings.llm = GoogleGenAI(model="gemini-2.5-flash")
Settings.embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-001")

# Small chunks = precise match, less context each; overlap saves split sentences
splitter = SentenceSplitter(chunk_size=128, chunk_overlap=20)

docs = [
    Document(text="Netsetos is an edtech company in Hyderabad offering GenAI, Data Science, and Cloud courses.", metadata={"source": "about.txt"}),
    Document(text="Refund policy: full refund within 7 days, 50% from 8-30 days, none after 30.", metadata={"source": "refund.txt"}),
    Document(text="The GenAI course costs 14,999 rupees with lifetime access and 5,000 GCP credits.", metadata={"source": "pricing.txt"}),
]

index = VectorStoreIndex.from_documents(docs, transformations=[splitter])

qa_prompt = PromptTemplate(
    "You are a Netsetos support assistant. Answer ONLY from the context.\n"
    "If it is not there, say you do not have that information.\n\n"
    "Context:\n{context_str}\n\nQuestion: {query_str}\nAnswer:"
)
engine = index.as_query_engine(similarity_top_k=2, text_qa_template=qa_prompt)

for q in ["What courses does Netsetos offer?", "Do you offer blockchain courses?"]:
    resp = engine.query(q)
    sources = [n.metadata.get("source", "?") for n in resp.source_nodes]
    print(f"Q: {q}\nA: {resp}\nsources: {sources}\n")

# Output:
# Q: What courses does Netsetos offer?  A: GenAI, Data Science, and Cloud.  sources: ['about.txt']
# Q: Do you offer blockchain courses?   A: I do not have that information.  sources: ['about.txt', 'pricing.txt']

#### 💡 What just happened

⚡ What just happened?The same 5-line skeleton, now with three knobs turned: a `SentenceSplitter(chunk_size=128)` as a transformation, a grounded `PromptTemplate` that forces "answer only from context", and `similarity_top_k=2`. **The grounding prompt is why "Do you offer blockchain courses?" returns "I do not have that information" instead of a hallucinated yes.** When to use a smaller chunk: precise factoid lookup (a policy line, a price); when to use larger: questions that need surrounding context. The tradeoff is always precision vs context.

- Drag the chunk-size slider and watch a real paragraph re-split into chunks live.

- Drag the overlap slider and see the shaded region each chunk shares with its neighbour grow.

- The gauge swings between *precision* (small chunks) and *context* (large chunks) as you drag.

- The chunk count and per-chunk token bars update on every move - the highest-leverage knob, made tangible.

Interactive: two sliders re-chunk the text live. Find the chunk size you would ship.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Chat Engine - Conversational RAG in One Line

### Chat Engine - Conversational RAG in One Line

Built-in memory + retrieval, no manual history list.

**A query engine forgets every question.** Conversational RAG needs history so "how much does *it* cost?" knows what "it" is. In 8.5 that meant a `MessagesPlaceholder` and a manual list you managed. LlamaIndex has a chat engine: `as_chat_engine(chat_mode="condense_plus_context")` gives memory, condensing, and retrieval in one parameter.

> 🦊 **Analogy**
>
> **The chat engine is a maitre-d with a good memory.** You do not re-introduce yourself each course; the maitre-d remembers you asked about the tasting menu, so "and how much is *it*?" is understood without you repeating "the tasting menu". `condense_plus_context` does exactly that - it condenses your follow-up plus the history into one standalone question, then retrieves and answers.
> **Where the analogy breaks down:** a maitre-d's memory is unbounded within a night; a chat engine's is not - very long conversations must be summarized or windowed, or the condensed query and cost balloon.

Turn 1: "What are the prerequisites?" Turn 2: "And how much does it cost?" How does `condense_plus_context` resolve "it"?

- Build the index as usual.

- **`as_chat_engine(chat_mode="condense_plus_context")`** is the whole feature - memory + condensing + retrieval.

- Call `.chat()` per turn; the engine carries history for you.

**📝 `03_chat_engine.py`** - *LlamaIndex*

In [ ]:
# LLAMAINDEX CHAT ENGINE - conversational RAG in one line
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

Settings.llm = GoogleGenAI(model="gemini-2.5-flash")
Settings.embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-001")

docs = [
    Document(text="Refund policy: full refund within 7 days, 50% from 8-30 days, none after 30."),
    Document(text="GenAI course: 14,999 rupees, 146 hours, 14 modules, Python + GCP."),
    Document(text="Prerequisites: basic Python and high-school math. No ML experience needed."),
]
index = VectorStoreIndex.from_documents(docs)

# One line: memory + condensing + retrieval
chat = index.as_chat_engine(chat_mode="condense_plus_context")

for q in ["What are the prerequisites?", "And how much does it cost?", "Can I get a refund after 2 weeks?"]:
    resp = chat.chat(q)
    print(f"You: {q}\nBot: {resp}\n")

# Output:
# You: What are the prerequisites?        Bot: Basic Python and high-school math; no ML needed.
# You: And how much does it cost?         Bot: 14,999 rupees.  ("it" resolved to the course)
# You: Can I get a refund after 2 weeks?  Bot: Yes - a 50% refund applies between days 8 and 30.

#### 💡 What just happened

⚡ What just happened?`condense_plus_context` condensed "how much does it cost?" plus the history into a standalone query ("how much does the GenAI course cost?"), retrieved the pricing document, and answered - all from one parameter. **LangChain (8.5) needed a MessagesPlaceholder and a history list you managed; LlamaIndex needs one argument.** When to use managed memory vs a manual list: reach for the chat engine for speed and simplicity, and drop to explicit history when you need to inspect or edit exactly what the model remembers.

---

## 🎯 Concept 4: The Data Model + Three-Way Comparison

### The Data Model + Three-Way Comparison

Document to Node to Index to QueryEngine - and when to pick which framework.

**Everything so far rests on one abstraction: the Node.** A `Document` is a whole source file; the splitter turns it into `Node`s, and a Node is a chunk of text plus its metadata plus its relationships (which document it came from, what came before and after). Retrieval and synthesis operate on Nodes, not raw strings. Understanding the Node is what makes the "5 lines" legible - and it frames the honest three-way choice between scratch, LangChain, and LlamaIndex.

> ⚛️ **Analogy**
>
> **The Node is the atom.** You cannot see or weigh a document directly in retrieval; the smallest thing LlamaIndex actually stores, scores, and returns is a Node - a chunk that remembers where it came from and who its neighbours are. Everything upstream (Document, loader) and downstream (Index, QueryEngine) exists to make and use Nodes.
> **Where the analogy breaks down:** atoms are indivisible, but a Node is not a fixed floor - it is only as small as the chunk size you picked, and you can even index tiny chunks for matching but return their larger parent for context, so "the atom" is really "the atom you chose when you set chunk size".

Your primary job is ingesting thousands of messy, table-heavy PDFs and querying across them. Which framework leads for that layer?

- Split one `Document` into `Node`s with a small chunk size.

- Inspect a Node: its `text`, inherited `metadata`, `node_id`, and `relationships`.

- Build the index directly from the nodes - the same atoms `from_documents` makes for you.

**📝 `04_data_model.py`** - *LlamaIndex*

In [ ]:
# THE DATA MODEL - Document -> Node -> Index -> QueryEngine
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

Settings.embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-001")

doc = Document(
    text="Netsetos GenAI course: 14,999 rupees for lifetime access, a Discord community, and 5,000 GCP credits. The course spans 14 modules over 146 hours, covering Python and math foundations, RAG systems, fine-tuning, tool use, agents, and LLMOps in production. Refund policy: full refund within 7 days, 50 percent from day 8 to day 30, and no refund after 30 days.",
    metadata={"source": "faq.txt"},
)

# A Document is split into Nodes: chunk + metadata + relationships
nodes = SentenceSplitter(chunk_size=64, chunk_overlap=10).get_nodes_from_documents([doc])
n = nodes[0]
print("node text :", n.text[:48])
print("metadata  :", n.metadata)               # inherited from the Document
print("relations :", list(n.relationships.keys()))  # SOURCE, NEXT, PREVIOUS...

index = VectorStoreIndex(nodes)                # build straight from nodes
print("chunks indexed:", len(nodes))

# Output:
# node text : Netsetos GenAI course: 14,999 rupees for lifetime
# metadata  : {'source': 'faq.txt'}
# relations : [<NodeRelationship.SOURCE>, <NodeRelationship.NEXT>]  (repr abbreviated)
# chunks indexed: 2

#### 💡 What just happened

⚡ What just happened?A Node is a chunk plus its metadata plus links to its source and neighbours - the atom LlamaIndex retrieves and synthesizes over. Now the framework choice is honest, not hype:

| Feature | From Scratch (8.1) | LangChain (8.5) | LlamaIndex (8.6) |
|---|---|---|---|
| Lines for basic RAG | ~60 | ~15 | ~5 |
| Chunking control | Full (your code) | Good (splitters) | Good (node parsers) |
| Document loaders | DIY | Large ecosystem | Large (LlamaHub) |
| Conversational RAG | DIY | Manual history | as_chat_engine() |
| Agent / orchestration | DIY | LangGraph | FunctionAgent / Workflows |
| Complex-PDF parsing | DIY | Third-party | LlamaParse |
| Best for | Learning, custom | Agents, deployment | RAG-first, data ingestion |

> 💡 **Info**
>
> When to use each - the tradeoff of control vs convenience
> - **Learning and custom control:** build from scratch (8.1) - you own every component.
> - **RAG-heavy, document-first apps:** LlamaIndex - cleaner abstractions, LlamaParse, caching ingestion.
> - **Agent-heavy, stateful apps:** LangChain + LangGraph - the orchestration ecosystem.
> - **Production (recommended):** the hybrid - LlamaIndex for data, LangChain for orchestration, joined by `as_langchain_tool()`.

## Go Deeper: The Data Specialist's Toolkit

Steps 5-10 are the depth LlamaIndex is famous for and a general framework rarely matches: LlamaParse, the caching IngestionPipeline, response-synthesis modes, decompose-and-route query engines, FunctionAgent orchestration, and the production hybrid. Each carries runnable code.

---

## 🎯 Concept 5: LlamaParse - GenAI-Native Document Parsing

### LlamaParse - GenAI-Native Document Parsing

Vision-language models for the PDFs that break every other parser. Four tiers.

**Real documents are hostile.** Scanned invoices, multi-column contracts, tables nested inside tables - a naive text extractor returns scrambled soup. LlamaParse uses vision-language models to read a page the way a human would, and you dial quality-vs-cost with a tier: **Fast** (1 credit/page), **Cost Effective** (3), **Agentic** (10, the default recommendation), **Agentic Plus** (45). New accounts get roughly 10,000 free credits a month.

> 🐟 **Analogy**
>
> **LlamaParse is the specialist fishmonger.** Anyone can fillet a simple fish, but a bony, awkward, half-frozen one needs a specialist who knows exactly where to cut. The tier is how much specialist attention you buy per page: Fast is a quick trim, Agentic Plus is the master breaking down the hardest catch bone by bone.
> **Where the analogy breaks down:** a fishmonger charges the same whatever the fish; LlamaParse charges by tier AND page, so on a 500-page report the tier choice is a real budget decision - and Fast cannot even produce markdown, so the cheapest option is not always available.

You need clean markdown (with tables) from a stack of scanned, table-heavy PDFs. Which tier can you NOT use?

- Create a `LlamaParse` parser and pick a tier via `parse_mode` / `result_type`.

- Pass it to `SimpleDirectoryReader` as a `file_extractor` for `.pdf`.

- The parsed documents flow into `VectorStoreIndex` exactly like any other.

**📝 `05_llamaparse.py`** - *LlamaParse*

In [ ]:
# LLAMAPARSE - GenAI-native parsing for the hardest PDFs
# pip install llama-cloud-services   (get a key at cloud.llamaindex.ai)
import os
from llama_cloud_services import LlamaParse
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

# Tier by cost/quality: Fast(1) / Cost Effective(3) / Agentic(10, default) / Agentic Plus(45) credits/page
parser = LlamaParse(
    api_key=os.environ["LLAMA_CLOUD_API_KEY"],
    result_type="markdown",               # Fast tier cannot do markdown - use Cost Effective+
    parse_mode="parse_page_with_agent",    # the Agentic tier: tables, charts, scanned pages
)

# Plug LlamaParse into the standard reader as a file_extractor
reader = SimpleDirectoryReader(
    input_files=["annual_report.pdf"],
    file_extractor={".pdf": parser},
)
docs = reader.load_data()
index = VectorStoreIndex.from_documents(docs)
print(f"parsed {len(docs)} document(s) with the Agentic tier (10 credits/page)")

# Output:
# parsed 1 document(s) with the Agentic tier (10 credits/page)
# (a vision-language model read the tables and charts into clean markdown)

#### 💡 What just happened

⚡ What just happened?LlamaParse is LlamaIndex's biggest data-layer differentiator - it reads image-heavy and table-heavy pages with vision-language models and plugs straight into `SimpleDirectoryReader`. The tier IS the cost/quality dial: most real-world PDFs want **Agentic**, prototypes want **Cost Effective**, and simple text-only pages can use **Fast** (but not for markdown). When to use it vs open-source: LlamaParse for scale and the hardest documents, Docling or PyMuPDF4LLM when you need free/self-hosted parsing and your PDFs are simpler.

- Toggle the document features - scanned? tables? charts? multi-column? - to describe your PDF.

- The widget recommends a tier and shows the credits/page it costs.

- Set a page count and watch the monthly cost compute against the 10,000 free-credit budget.

- A quality bar rises with the tier - see the cost/quality dial as a live tradeoff.

Interactive: toggle features + page count, get a recommended tier and a real monthly cost.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: IngestionPipeline - Declarative Processing with Caching

### IngestionPipeline - Declarative Processing with Caching

A transformations list that hashes its work, so re-runs skip what did not change.

**Re-indexing is where naive RAG bleeds money.** If you delete and rebuild the index every night, ingesting 10,000 documents costs 10,000 embeddings - even when only 50 changed. The `IngestionPipeline` is a declarative list of transformations (splitter, embedder, extractors) that *caches per node+transformation hash*: a re-run recomputes only the documents whose content changed. Nightly re-indexing goes from O(N) to O(changed).

> 📋 **Analogy**
>
> **The pipeline is a prep list you run once and cache.** A good kitchen does not re-prep every ingredient each service - it preps once, labels each container, and reuses what is still good; only the items that spoiled or changed get re-prepped. The hash is the label: same input, same transform, same result, so skip it.
> **Where the analogy breaks down:** prepped food spoils on a timer; a cache entry is invalidated only by a content change (the hash), so a document that never changes is never re-embedded, no matter how many nightly runs pass over it.

Your pipeline re-runs nightly on 10,000 docs; ~50 change per day. With caching on, how many get re-embedded?

- Build an `IngestionPipeline` from a list of transformations.

- Run it, then `persist()` the node+transform hash cache.

- Change one document and re-run: the two unchanged docs are cache hits.

**📝 `06_ingestion_pipeline.py`** - *LlamaIndex*

In [ ]:
# INGESTIONPIPELINE - declarative processing with hash caching
from llama_index.core import Document
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

docs = [
    Document(text="Refund policy: full refund within 7 days.", doc_id="refund"),
    Document(text="The GenAI course costs 14,999 rupees.", doc_id="pricing"),
    Document(text="Live classes run daily at 7 PM IST.", doc_id="schedule"),
]

pipeline = IngestionPipeline(transformations=[
    SentenceSplitter(chunk_size=128, chunk_overlap=10),
    GoogleGenAIEmbedding(model_name="gemini-embedding-001"),
])

nodes = pipeline.run(documents=docs)      # first run: all 3 processed + embedded
pipeline.persist("./pipeline_cache")      # save the node+transform hash cache
print("first run  - processed:", len(nodes))

# Later: only one document changed
docs[1].text = "The GenAI course costs 14,999 rupees, lifetime access included."
pipeline.load("./pipeline_cache")
nodes = pipeline.run(documents=docs)      # the 2 unchanged docs are cache hits
print("second run - only the changed doc was reprocessed (2 cache hits)")

# Output:
# first run  - processed: 3
# second run - the 2 unchanged docs hit the cache (embeddings reused, not recomputed)

#### 💡 What just happened

⚡ What just happened?The `IngestionPipeline` replaces the manual loader-splitter-embed-store pattern with a single declarative object, and the caching is the real production win: each node+transformation gets a hash, so re-running on 10,000 documents when 50 changed reprocesses only 50. **Use `filename_as_id=True` in SimpleDirectoryReader for stable IDs so the cache actually hits across runs**, and a `RedisKVStore` cache + `docstore` for distributed dedup. When to use it vs a plain re-index: the moment your corpus is big and mostly-stable, the cache pays for itself.

- A grid of document tiles represents your corpus.

- Drag the "how many changed" slider to mark some tiles as edited.

- Press "Re-run pipeline": unchanged tiles flash "cache hit" and skip; changed tiles reprocess.

- The scoreboard tallies reprocessed vs skipped and the cost/time saved - O(changed) vs O(N), live.

Interactive: set how many docs changed, hit Re-run, watch the cache save the rest.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Response Synthesis Modes - compact, refine, tree_summarize

### Response Synthesis Modes - compact, refine, tree_summarize

Same retrieved nodes, different synthesizer. Very different call counts.

**Retrieval gets the right nodes; synthesis turns them into an answer - and how you synthesize is a real dial.** `compact` (the default) stuffs as many nodes as fit into 1-2 LLM calls. `refine` walks the nodes one at a time, refining the answer at each (N calls). `tree_summarize` summarizes nodes in pairs up a tree (log N calls). Same retrieval, wildly different latency, cost, and quality - and most people never touch it.

> 🍝 **Analogy**
>
> **Three ways to cook the same fish.** Sear one piece hot and fast (compact - one confident pass, but too much fish overflows the pan). Simmer each piece in sequence, adjusting as you go (refine - thorough, but the first piece flavours everything after it). Build a tower, combining pairs upward (tree_summarize - balanced, no single piece dominates, but more steps). Same ingredient, three techniques, three results.
> **Where the analogy breaks down:** a chef tastes and adapts; the synthesis mode is fixed per query - you choose the technique up front by the shape of the task (factoid vs summary), not by tasting the half-cooked answer.

You need to summarize 30 retrieved chunks without letting the first chunk dominate the answer. Which mode?

- Build the index once.

- Create three query engines that differ only in `response_mode`.

- Run the same query through each and compare the answer and call pattern.

**📝 `07_synthesis_modes.py`** - *LlamaIndex*

In [ ]:
# RESPONSE SYNTHESIS MODES - same retrieval, different synthesizer
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

Settings.llm = GoogleGenAI(model="gemini-2.5-flash")
Settings.embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-001")

docs = [Document(text=t) for t in [
    "Module 1 covers Python and math foundations.",
    "Module 8 covers RAG systems end to end.",
    "Module 14 covers LLMOps and evaluation.",
]]
index = VectorStoreIndex.from_documents(docs)

# Same nodes, three synthesizers - different call patterns
for mode in ["compact", "refine", "tree_summarize"]:
    engine = index.as_query_engine(response_mode=mode, similarity_top_k=3)
    resp = engine.query("Summarize what the course covers.")
    print(f"[{mode:15s}] {str(resp)[:64]}")

# Output:
# [compact        ] The course covers Python foundations, RAG, and LLMOps.   (1-2 LLM calls)
# [refine         ] ... refines the answer chunk by chunk in sequence         (N LLM calls)
# [tree_summarize ] ... merges chunk summaries pairwise up a tree             (log N LLM calls)

#### 💡 What just happened

⚡ What just happened?One argument, `response_mode`, changed how the SAME retrieved nodes become an answer. **Switching from compact to tree_summarize can transform summarization quality overnight** - and it is the most underappreciated dial in RAG. When to use which: `compact` for fast factoid Q and A (1-2 calls); `refine` when every chunk genuinely matters and you accept N calls plus first-chunk bias; `tree_summarize` for summaries where order should not dominate, at log N calls. The cost scales with the mode, so choose deliberately, not by default.

- Six retrieved chunks sit at the top, ready to synthesize.

- Click compact: watch them merge into 1-2 LLM calls.

- Click refine: watch the answer walk chunk by chunk (N calls), with the first-chunk-bias warning.

- Click tree_summarize: watch a log(N) merge tree build - and read the live call-count and cost estimate for each.

Interactive: toggle the three modes, watch the LLM-call pattern and cost change over the same chunks.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 8: Advanced Query Engines - Router, SubQuestion, Citation

### Advanced Query Engines - Router, SubQuestion, Citation

Route by intent, decompose comparisons, cite sources - built in, not hand-rolled.

**A single query engine answers single questions.** "Compare Company A's Q3 revenue with Company B's" is really two questions plus a synthesis - and it may span two separate indexes. The `SubQuestionQueryEngine` decomposes it automatically: it generates sub-questions, routes each to the right `QueryEngineTool`, and synthesizes a comparative answer. Alongside it: `RouterQueryEngine` (pick one engine per query) and `CitationQueryEngine` (inline [1][2] source attribution).

> 🧩 **Analogy**
>
> **A comparison is several questions wearing one coat.** Ask a good analyst to "compare A and B" and they do not answer in one breath - they look up A, look up B, then compare. SubQuestion makes that reflex automatic: split, fetch each part from the right source, then combine.
> **Where the analogy breaks down:** a human knows when a question is really one question; SubQuestion decomposes based on the tool descriptions you write, so a vague description ("Company A data") routes worse than a specific one ("Company A Q3 financials and revenue"). The routing is only as good as your metadata.

"Compare the Q3 revenue of Company A vs Company B", with A and B in separate indexes. Which engine?

- Build two indexes and wrap each as a `QueryEngineTool` with a descriptive name.

- `SubQuestionQueryEngine.from_defaults` takes those tools.

- One comparison query is split, routed per sub-question, and synthesized.

**📝 `08_subquestion.py Query`** - *Engines*

In [ ]:
# SUBQUESTIONQUERYENGINE - decompose a comparison into sub-questions
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.tools import QueryEngineTool
from llama_index.core.query_engine import SubQuestionQueryEngine
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

Settings.llm = GoogleGenAI(model="gemini-2.5-flash")
Settings.embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-001")

# Two separate indexes, one per company
idx_a = VectorStoreIndex.from_documents([Document(text="Company A Q3 revenue was 120 crore, up 12%.")])
idx_b = VectorStoreIndex.from_documents([Document(text="Company B Q3 revenue was 95 crore, up 20%.")])

tools = [
    QueryEngineTool.from_defaults(idx_a.as_query_engine(),
        name="company_a", description="Q3 financials and revenue for Company A"),
    QueryEngineTool.from_defaults(idx_b.as_query_engine(),
        name="company_b", description="Q3 financials and revenue for Company B"),
]

engine = SubQuestionQueryEngine.from_defaults(query_engine_tools=tools)
print(engine.query("Compare the Q3 revenue of Company A and Company B."))

# Output:
# What the engine does internally (illustrative):
#   "What is Company A Q3 revenue?" -> company_a
#   "What is Company B Q3 revenue?" -> company_b
# Synthesized: A earned 120 crore (up 12%), B 95 crore (up 20%); A is higher, B grew faster.

#### 💡 What just happened

⚡ What just happened?`SubQuestionQueryEngine` split "compare A vs B" into two sub-questions, routed each to the matching index tool, ran them, and synthesized one comparative answer - multi-hop retrieval you would otherwise hand-build. **The routing quality lives in the tool descriptions: "Q3 financials and revenue for Company A" routes far better than "Company A data".** When to use which: `SubQuestion` for comparisons and multi-source questions; `Router` when one query should go to exactly one of several engines; `Citation` when compliance demands inline source attribution.

- Pick a comparison query from the dropdown (revenue, headcount, growth...).

- Watch it decompose into one sub-question per source.

- See each sub-question route to the matching QueryEngineTool and return its answer.

- The engine then synthesizes the parts into one comparative answer - the whole SubQuestion flow, interactive.

Interactive: choose a query and watch it split, route, and recombine.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 9: Workflows and Agents - FunctionAgent, AgentWorkflow

### Workflows and Agents - FunctionAgent, AgentWorkflow

An agent is a query engine that decides which tool to call. And agents nest as tools.

**When the path must be chosen at runtime, you need an agent.** LlamaIndex's `FunctionAgent` gives the LLM a set of tools - a query engine, a calculator, a web search - and lets it pick, using the model's native function-calling. It is the preferred agent for capable models because function-calling is more reliable than ReAct's Thought/Action/Observation prompting. `AgentWorkflow` composes several agents with `can_handoff_to` for multi-agent systems.

> 🤖 **Analogy**
>
> **An agent is a specialist team lead who delegates.** Given a task, the lead does not do everything personally - they decide which specialist (tool) handles each part and hand off. And because a whole team can be wrapped as a single "specialist", agents nest as tools: a research agent becomes one tool for a writing agent.
> **Where the analogy breaks down:** a human lead remembers yesterday; a `FunctionAgent` is stateless by default - it forgets between runs unless you give it a `Context`, so persistent memory is opt-in, not automatic.

You are building an agent on a capable function-calling model (Gemini 2.5). Which LlamaIndex agent is the production default?

- Wrap a query engine as a `QueryEngineTool` and a function as a `FunctionTool`.

- `FunctionAgent(tools, llm, system_prompt)` lets the LLM choose which to call.

- Agents are async - an agent may fire several model and tool calls per turn, so you `await agent.run(...)` inside an `async def` and launch it with `asyncio.run()`. This is the module’s first async code; the pattern is always wrap in `async def`, launch with `asyncio.run()`.

**📝 `09_function_agent.py`** - *FunctionAgent*

In [ ]:
# FUNCTIONAGENT - the LLM decides which tool to call (async, function-calling)
import asyncio
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.tools import QueryEngineTool, FunctionTool
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

Settings.embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-001")

index = VectorStoreIndex.from_documents([Document(text="Refund policy: full refund within 7 days.")])
policy_tool = QueryEngineTool.from_defaults(
    index.as_query_engine(), name="policy_search",
    description="Search Netsetos policies, pricing, and schedules.")

def add_days(start_day: int, days: int) -> int:
    """Add a number of days to a start day."""
    return start_day + days

agent = FunctionAgent(
    tools=[policy_tool, FunctionTool.from_defaults(add_days)],
    llm=GoogleGenAI(model="gemini-2.5-flash"),
    system_prompt="Answer Netsetos questions using the tools.",
)

async def main():
    print(await agent.run("What is the refund policy?"))

asyncio.run(main())

# Output:
# The agent calls policy_search -> "Full refund within 7 days." -> answers:
# You can get a full refund within 7 days of purchase.

#### 💡 What just happened

⚡ What just happened?The `FunctionAgent` read the tool descriptions, chose `policy_search` for a policy question, ran it, and answered - you wrote no router. **FunctionAgent uses native function-calling and is the production default for capable LLMs; `ReActAgent` (Thought/Action/Observation) is the fallback for models without reliable tool-calling.** Because a `QueryEngineTool` wraps any engine, and agents are themselves query engines, agents nest as tools - which is how `AgentWorkflow` builds multi-agent systems with `can_handoff_to`. The tradeoff is memory: FunctionAgent is stateless by default, so add a `Context` when you need it to remember across runs.

---

## 🎯 Concept 10: Production + LlamaIndex vs LangChain + Multilingual

### Production + LlamaIndex vs LangChain + Multilingual

Persist, evaluate, go multilingual, and combine both frameworks.

**Shipping LlamaIndex needs four things a demo skips:** real persistence (an external vector store, not in-memory), evaluation before deploy (a `FaithfulnessEvaluator` catches hallucinations), multilingual reach (swap the embedder to BGE-M3), and - the recommended architecture - the LangChain hybrid, where a LlamaIndex query engine becomes a LangChain tool via one adapter call.

> 🧰 **Analogy**
>
> **Production is the shared bench.** The sushi knife (LlamaIndex) and the workshop (LangChain) sit side by side: the knife preps the data, the workshop runs the service. You do not choose one and throw the other out; `QueryEngineTool.as_langchain_tool()` is the handle that lets the workshop pick up the knife's work.
> **Where the analogy breaks down:** a shared bench is free; combining two frameworks is not - it means two dependency trees and glue code, so "use both" is justified only when each is genuinely better at its layer.

What is the recommended 2026 production architecture combining LlamaIndex and LangChain?

- **Persist + reload** the index (external stores auto-persist in real prod).

- **Evaluate** for hallucination with a `FaithfulnessEvaluator`.

- **Multilingual**: swap the embedder to `BAAI/bge-m3`.

- **Hybrid**: expose the engine as a LangChain tool.

**📝 `10_production.py`** - *Production*

In [ ]:
# PRODUCTION LLAMAINDEX - persist, evaluate, multilingual, the LangChain hybrid
from llama_index.core import (VectorStoreIndex, Document, StorageContext,
                             load_index_from_storage, Settings)
from llama_index.core.evaluation import FaithfulnessEvaluator
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

Settings.llm = GoogleGenAI(model="gemini-2.5-flash")
Settings.embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-001")

# 1. Persist and reload (external vector stores auto-persist in real production)
index = VectorStoreIndex.from_documents([Document(text="Refund within 7 days.")])
index.storage_context.persist("./storage")
index = load_index_from_storage(StorageContext.from_defaults(persist_dir="./storage"))

# 2. Evaluate for hallucination before shipping (full harness is Lesson 8.11)
resp = index.as_query_engine().query("What is the refund window?")
print("faithful:", FaithfulnessEvaluator().evaluate_response(response=resp).passing)

# 3. Multilingual: swap the embedder to BGE-M3 (100+ languages, incl. Hindi)
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-m3")  # set BEFORE building a fresh multilingual index

# 4. The hybrid: expose a LlamaIndex engine as a LangChain tool for a LangGraph agent
# from llama_index.core.tools import QueryEngineTool
# lc_tool = QueryEngineTool.from_defaults(index.as_query_engine()).as_langchain_tool()
print("faithful check + BGE-M3 multilingual + LangChain hybrid ready")

# Output:
# faithful: True
# faithful check + BGE-M3 multilingual + LangChain hybrid ready

#### 💡 What just happened

⚡ What just happened?Production LlamaIndex persists to an external store, evaluates with a `FaithfulnessEvaluator` before deploy, goes multilingual by swapping the embedder to `BAAI/bge-m3` (Hindi and 100+ languages), and joins LangChain via `as_langchain_tool()`. **The recommended architecture is the hybrid: LlamaIndex owns the data layer (LlamaParse, IngestionPipeline, synthesis), LangChain owns orchestration (agents, state, LangSmith).** The tradeoff of in-memory vs external: the moment you have real traffic, an external vector store (Qdrant, Pinecone) is the first production change, and it persists automatically.

## Putting It Together

You built RAG in 5 lines, then met the data-specialist toolkit a general framework rarely matches: LlamaParse, caching ingestion, synthesis modes, decompose-and-route query engines, and FunctionAgent orchestration. The through-line: LlamaIndex trades LangChain's breadth for depth on one job, and in production the two compose - the knife and the workshop on one bench.

> 📦 **Info**
>
> 🔑 Key takeaways
> - **RAG in 5 lines.** `VectorStoreIndex.from_documents` hides chunk+embed+store; `as_query_engine` hides retrieve+synthesize - understand the 8.1 build first.
> - **The Node is the atom.** A Node is a chunk plus metadata plus links; chunk size decides what a unit of retrieval even is.
> - **The data toolkit is the differentiator.** LlamaParse for hard PDFs, the caching IngestionPipeline for O(changed) re-indexing, and response-synthesis modes for the same-retrieval-different-answer dial.
> - **Decompose-and-route.** SubQuestion splits comparisons across indexes; Router picks one engine; Citation adds source attribution.
> - **FunctionAgent over ReActAgent** for capable models; agents nest as tools, and AgentWorkflow adds multi-agent handoff.
> - **Specialize, then compose.** LlamaIndex for data, LangChain for orchestration, joined by `as_langchain_tool()`. Use the current stack: `GoogleGenAI`, `Settings` (not ServiceContext), `gemini-2.5-flash`.

> ✅ **Info**
>
> 🗺️ Where this goes next
> - In Lesson 8.7 (**GraphRAG**) the routing and decomposition you did in step 8 extend to retrieval over a knowledge graph (PropertyGraphIndex).
> - In Lesson 8.10 (**Agentic RAG**) step 9's FunctionAgent grows into a full multi-agent system with planning and memory.
> - In Lesson 8.11 (**RAG Evaluation**) step 10's FaithfulnessEvaluator becomes a golden-set + CI harness.

*Practice exercises are in the companion practice notebook.*

Eight exercises, easy to hard. Each builds or operates one layer of the LlamaIndex data toolkit.

---

## 🎓 Summary

You've completed the practical part of **RAG with LlamaIndex- The Data Framework**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-8_6.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-8.6.html` - regenerate this notebook whenever the source HTML is updated.*
